# 02 — Preprocessing: UniGaze face normalisation

**Why**: UniGaze was trained on *normalised* faces (Zhang, Sugano & Bulling, ETRA 2018): the image is warped as if
a virtual camera looked straight at the face centre from 600 mm with focal length 960 px, removing head roll and
scale. The capture app's 224×224 crops are plain bounding-box crops, so they differ in scale, roll and perspective
from what the network saw during pre-training.

**What is approximate**: your webcam was never calibrated. As in UniGaze's own in-the-wild video script
(`predict_gaze_video.py`), the camera is approximated: crop 2× the face box, focal length = 4 × crop width,
no lens distortion. This approximation is recorded in every cache and experiment (`preprocessing.version`). It is
identical for train / validation / test / deployment, so it does not create a train–test mismatch; it only means
UniGaze's angles are not physically exact, which is why they are used as *features*, never as labels.

**Steps**: raw frame → YuNet 5-point landmarks → solvePnP head pose → normalisation warp → RGB uint8 cache.
No flipping, rotation, cropping jitter or grayscale conversion is ever applied.

**Outputs**: `processed_cache/<version>/{train,validation}_faces_uint8.npy` (≈2.3 GB + 0.75 GB),
`*_geometry.parquet`, `*_cache_info.json`. The test cache is built only in notebook 06.

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

In [ ]:
# Stop early if the audit has not passed.
from src.dataset import require_audit_pass, load_split
require_audit_pass(cfg)
train_df, val_df = load_split(cfg, "train"), load_split(cfg, "validation")
print(len(train_df), "train frames |", len(val_df), "validation frames")
print("Preprocessing settings:", dict(cfg["preprocessing"]))
print("YuNet model:", cfg.paths.yunet_model, "exists:", cfg.paths.yunet_model.is_file())

## 1. Look at the transformation on a few frames (cheap)
Left: raw webcam frame (unchanged). Middle: capture-app face crop. Right: UniGaze-normalised face.
**What could go wrong**: if the right image is upside-down, mirrored or shows only part of the face, the landmarks
were mismatched — stop and report it. Expected: an upright, centred face of similar size in every example.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.preprocessing import YuNetLandmarker, normalize_face, read_bgr, read_face_crop_rgb

det = YuNetLandmarker(cfg.paths.yunet_model)
rng = np.random.default_rng(0)
examples = train_df.groupby("participant_id").sample(1, random_state=0).reset_index(drop=True)
fig, axes = plt.subplots(len(examples), 3, figsize=(10, 3.2 * len(examples)))
for i, r in examples.iterrows():
    raw = read_bgr(cfg.paths.dataset_root / r.raw_path)
    lm = det.detect(raw, expected_eyes=(r.left_eye_center, r.right_eye_center))
    res = normalize_face(raw, lm, dict(cfg["preprocessing"])) if lm is not None else None
    axes[i, 0].imshow(raw[:, :, ::-1]); axes[i, 0].set_title(f"{r.sample_id} raw", fontsize=8)
    if lm is not None:
        axes[i, 0].scatter(lm.points[:, 0], lm.points[:, 1], s=8, c="lime")
    axes[i, 1].imshow(read_face_crop_rgb(cfg.paths.dataset_root / r.face_path)); axes[i, 1].set_title("capture crop", fontsize=8)
    if res is not None and res.ok:
        axes[i, 2].imshow(res.image_rgb)
        axes[i, 2].set_title(f"normalised (head pitch/yaw {np.degrees(res.head_pitchyaw_norm).round(1)} deg)", fontsize=8)
    else:
        axes[i, 2].set_title(f"FAILED: {None if res is None else res.reason}", fontsize=8)
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Success rate on a small sample (before committing to the full cache)
Normalisation can fail when no face is found, solvePnP fails, or the normalised head angle exceeds 80°.
Failed frames are *not* used for training or scored as predictions; they are reported as tracking failures.
A success rate below ~98 % deserves inspection before the full build.

In [ ]:
from collections import Counter
sample = train_df.sample(min(200, len(train_df)), random_state=1)
reasons = Counter()
for r in sample.itertuples():
    raw = read_bgr(cfg.paths.dataset_root / r.raw_path)
    lm = det.detect(raw, expected_eyes=(r.left_eye_center, r.right_eye_center))
    reasons["no_face" if lm is None else (normalize_face(raw, lm, dict(cfg["preprocessing"])).reason or "ok")] += 1
print(reasons, f"-> success {reasons['ok'] / len(sample):.1%}")

## 3. Build the normalised cache — CPU intensive, opt-in
Set `RUN_CACHE_BUILD = True` to build. It uses `hardware.num_workers` processes (Windows *spawn* is supported
because the worker functions live in `src/preprocessing.py`). It is **resumable**: interrupting and rerunning
continues where it stopped. If workers misbehave in Jupyter on Windows, either set `NUM_WORKERS = 0` or run
`python scripts/build_cache.py --split train` from a terminal.

Expected time: roughly 15–40 minutes for 30,000 training frames, depending on disk and CPU.

In [ ]:
RUN_CACHE_BUILD = False            # <- set True deliberately
NUM_WORKERS = cfg["hardware"]["num_workers"]

from src.pipeline import build_cache_for_split
if RUN_CACHE_BUILD:
    for split in ("train", "validation"):
        info = build_cache_for_split(cfg, split, num_workers=NUM_WORKERS)
        print(split, info["normalized_ok"], "/", info["rows"], "ok; failures:", info["failures"])
else:
    print("Cache build skipped (RUN_CACHE_BUILD = False).")

## 4. Verify the cache
* every manifest row is present, in the same order;
* failure count and reasons;
* the cached pixels equal a fresh recomputation from the stored geometry (proves the cache and the live
  deployment path produce identical inputs).

In [ ]:
from src.pipeline import split_source
from src.preprocessing import warp_with_stored_geometry
for split in ("train", "validation"):
    df, src, geo = split_source(cfg, split)
    print(f"{split}: {int(src.ok.sum())}/{len(df)} normalised; failures:",
          geo.loc[~geo.norm_ok, "norm_reason"].value_counts().to_dict())
    for pos in np.random.default_rng(0).choice(np.where(src.ok)[0], 5, replace=False):
        g = geo.iloc[pos]
        Wm = g[[f"W{i}" for i in range(9)]].to_numpy(float)
        again = warp_with_stored_geometry(read_bgr(cfg.paths.dataset_root / df.raw_path.iloc[pos]),
                                          [g.crop_x0, g.crop_y0, g.crop_x1, g.crop_y1], Wm)
        assert np.array_equal(again, src(pos)), "cache differs from recomputation"
print("Cache verified.")

## 5. Batch visualisation (exactly what the model will receive, before ImageNet normalisation)
Check: faces upright and centred, colours natural (RGB, not BGR-blue skin), no flips.

In [ ]:
import torch
from src.dataset import GazeFrameDataset, collate
df, src, _ = split_source(cfg, "train")
ds = GazeFrameDataset(df, src)
batch = collate([ds[i] for i in np.random.default_rng(2).choice(np.where(src.ok)[0], 16, replace=False)])
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, img, t in zip(axes.ravel(), batch["image"], batch["target"]):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
    ax.set_title(f"({t[0]*W:.0f},{t[1]*H:.0f})px", fontsize=7)
plt.suptitle("Training inputs with their screen targets"); plt.show()
print("tensor:", batch["image"].shape, batch["image"].dtype, "-> ImageNet normalisation happens on the GPU")